# 12장 공공데이터 API: 실제 자료를 받아 그리기

『AI 시대의 파이썬 데이터 시각화』 예제 노트북입니다. 셀 첫 줄의 번호(P·C·B)는 책 지면의 코드 번호와 같습니다. 아래 안내에 따라 실행합니다.

**키가 없거나 C05가 오류로 멈추면** C01–C05와 C11은 지면의 출력으로 읽고, C06부터 예제 저장소의 저장본(받아 둔 CSV, 12.3절)으로 실행합니다.

## 준비

새 노트북이면 이 장의 코드보다 먼저 아래 준비 셀을 실행합니다(책의 앞 장과 같은 셀).

**코랩이면 이 셀(P02)을, 내 컴퓨터면 다음 셀(P03)을 실행합니다.** 둘 다 실행하지 않습니다.

In [ ]:
# P02 | 한글 글꼴(코랩): 나눔글꼴을 설치하고 Matplotlib에 등록합니다.
!apt-get -qq update
!apt-get -qq install -y fonts-nanum

import matplotlib.pyplot as plt
from matplotlib import font_manager

for path in font_manager.findSystemFonts():
    if 'Nanum' in path:
        font_manager.fontManager.addfont(path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

**내 컴퓨터(Mac·Windows)면 이 셀을 실행합니다.** Windows는 글꼴 이름을 `'Malgun Gothic'`으로 바꿉니다.

In [ ]:
# P03 | 한글 글꼴(Mac·Windows): 그래프를 그리기 전에 한 번 실행합니다.
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'AppleGothic'   # Windows는 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False    # 음수 기호가 깨지지 않게

## 12.1 API와 인증키

In [ ]:
# P07 | API 준비: requests 버전을 보고 키를 코드 밖에서 읽습니다.
import tomllib
from pathlib import Path
from urllib.parse import unquote

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

print('requests', requests.__version__)


def get_key(name='DATA_GO_KR_KEY'):
    try:                                   # 코랩: 보안 비밀
        from google.colab import userdata
        key = userdata.get(name)
    except ImportError:                    # 내 컴퓨터: secrets.toml
        path = Path('.streamlit/secrets.toml')
        text = path.read_text(encoding='utf-8') if path.exists() else ''
        key = tomllib.loads(text).get(name)
    except Exception:                      # 코랩: 키 없음·액세스 꺼짐
        key = None
    return unquote(key.strip()) if key else None   # 앞뒤 공백 제거


key = get_key()
print('키:', '있음' if key else '없음 → C06부터 실행')

**키가 없거나 C05가 오류로 멈추면** C01–C05와 C11은 지면의 출력으로 읽고, C06부터 예제 저장소의 저장본(받아 둔 CSV, 12.3절)으로 실행합니다.

## 12.2 요청과 응답

In [ ]:
# C01 | 첫 요청: 서울의 2025년 1월 1~3일 일자료를 JSON으로 받습니다.
URL = ('https://apis.data.go.kr/1360000/'
       'AsosDalyInfoService/getWthrDataList')
params = {'serviceKey': key, 'pageNo': 1, 'numOfRows': 10,
          'dataType': 'JSON', 'dataCd': 'ASOS', 'dateCd': 'DAY',
          'startDt': '20250101', 'endDt': '20250103',
          'stnIds': '108'}
r = requests.get(URL, params=params, timeout=10)
print(r.status_code, r.headers['Content-Type'])
print(r.text[:71])

In [ ]:
# C02 | 응답 읽기: 결과 코드를 먼저 보고 몸통에서 항목 목록을 꺼냅니다.
res = r.json()['response']
print(res['header'])
if res['header']['resultCode'] != '00':      # 00이 아니면 여기서 멈춤
    raise RuntimeError(res['header']['resultMsg'])
body = res['body']
items = body['items']['item']
print('전체', body['totalCount'], '건 · 받은 항목', len(items), '건')
print(items[0]['tm'], items[0]['stnNm'], items[0]['avgTa'])

## 12.3 형식 바꾸기와 점검

In [ ]:
# C03 | 표로 바꾸기: 항목 목록을 표로 만들고 쓸 열만 남깁니다.
cols = ['tm', 'stnNm', 'avgTa', 'minTa', 'maxTa', 'sumRn', 'avgRhm']
day3 = pd.DataFrame(items)[cols]        # 세 날의 표
print(len(items[0]), '개 필드 중', len(cols), '개')
print(day3)
print(day3['minTa'].tolist(), '→ 최솟값', day3['minTa'].min())

In [ ]:
# C04 | 형식 바꾸기: 숫자·날짜 형식을 바꾸고 빈 강수량을 0으로 채웁니다.
def fix_types(df):
    df = df.copy()                         # 받은 표는 두고 사본을 고침
    for col in ['avgTa', 'minTa', 'maxTa', 'sumRn', 'avgRhm']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['tm'] = pd.to_datetime(df['tm'])
    df['sumRn'] = df['sumRn'].fillna(0)    # 빈 강수량은 0으로(본문 참고)
    return df


day3 = fix_types(day3)
print(day3.dtypes)
print('최저 기온의 최솟값:', day3['minTa'].min())

In [ ]:
# C05 | 1년 받기: 지점마다 1년을 받아 확인하고 저장본으로 남깁니다.
stations = {'108': '서울', '159': '부산', '105': '강릉', '184': '제주'}


def fetch_daily(stn):
    p = dict(params, stnIds=stn, numOfRows=366,   # 윤년도 한 쪽에
             startDt='20250101', endDt='20251231')
    r = requests.get(URL, params=p, timeout=10)
    if r.status_code != 200:               # 키 오류는 401·403
        raise RuntimeError(f'{r.status_code} {r.text[:80]}')
    res = r.json()['response']
    if res['header']['resultCode'] != '00':
        raise RuntimeError(res['header']['resultMsg'])
    items = res['body']['items']['item']
    if len(items) != res['body']['totalCount']:
        raise RuntimeError(f'{stn}: {len(items)}건만 받음')
    return fix_types(pd.DataFrame(items)[cols])


weather = pd.concat([fetch_daily(stn) for stn in stations],
                    ignore_index=True)
weather.to_csv('asos_2025_daily.csv', index=False)
print(len(weather), '행')

In [ ]:
# C06 | 저장본: 키가 없거나 C05가 멈췄으면 예제 저장소의 저장본을 읽습니다.
if 'weather' not in globals():          # C05가 표를 만들지 못했으면
    weather = pd.read_csv('asos_2025_daily.csv', parse_dates=['tm'])
print(len(weather), '행 ·', weather['tm'].dtype)

In [ ]:
# C07 | 점검: 지점마다 행 수와 기간을 보고 결측·중복을 셉니다.
print(weather.groupby('stnNm')['tm'].agg(['size', 'min', 'max']))
print('결측이 있는 열:', weather.columns[weather.isna().any()].tolist())
same_day = weather.duplicated(['stnNm', 'tm'])   # 같은 지점·같은 날
print('중복:', same_day.sum(), '행')

### 직접 해 보기

**W12-01** `weather`(키가 없으면 저장본)에서 강릉이나 제주를 골라 점검합니다.

1. 고른 지점의 행만 골라 C07처럼 행 수와 첫날·마지막 날을 출력합니다.
2. 일강수량이 0보다 큰 날과 그렇지 않은 날(0.0mm로 적힌 날 포함)을 셉니다.
3. 일강수량이 가장 많은 날과 그 양을 `idxmax()`(6.4절 C08)로 찾습니다.

**완료 기준**: 고른 지점의 행 수와 기간을 출력했고, 두 가지 날의 합이 행 수와 같으며, 일강수량이 가장 많은 날을 찾았다.

In [ ]:
# W12-01 — 여기에서 직접 해 봅니다.

## 12.4 질문에 맞는 차트 셋

In [ ]:
# C08 | 월 요약표: 기온은 평균, 강수량은 합계로 한 달·한 도시씩 줄입니다.
two = weather[weather['stnNm'].isin(['서울', '부산'])].copy()
two['month'] = two['tm'].dt.month
by_month = two.groupby(['month', 'stnNm'], as_index=False).agg(
    temp=('avgTa', 'mean'), rain=('sumRn', 'sum'))
print(len(by_month), '행')
print(by_month.round(1).head(4))

In [ ]:
# C09 | 완성 그림: 위는 월평균 기온, 아래는 월 강수량 합계입니다.
city_colors = {'서울': '#0072B2', '부산': '#D55E00'}
looks = {'서울': ('-', 'o', ''), '부산': ('--', 's', '//')}
fig, (top, bottom) = plt.subplots(2, 1, figsize=(6, 2.3), sharex=True,
                                  layout='constrained')
for i, city in enumerate(['서울', '부산']):
    part = by_month[by_month['stnNm'] == city]
    line, marker, hatch = looks[city]
    top.plot(part['month'], part['temp'], color=city_colors[city],
             linestyle=line, marker=marker, label=city)
    bottom.bar(part['month'] - 0.2 + 0.4 * i, part['rain'], width=0.4,
               color=city_colors[city], hatch=hatch,
               edgecolor='black', linewidth=0, label=city)
top.axhline(0, color='#666666', linewidth=0.8)
top.set(ylabel='기온 (℃)', title='2025년 서울과 부산의 날씨')
top.legend(loc='upper left', ncols=2, frameon=False)
bottom.legend(loc='center left', ncols=2, frameon=False)  # 1~3월 빈자리
bottom.set(xlabel='월', ylabel='강수량 (mm)', xticks=range(1, 13))
source = '자료: 기상청 ASOS 일자료(공공데이터포털, 2025년 서울·부산)'
bottom.text(0.01, 0.98, source, transform=bottom.transAxes,
            va='top', fontsize=9, color='#666666')
plt.show()

In [ ]:
# C09B | 그림 검산: 그림에서 읽을 값을 요약표에서 다시 꺼냅니다.
temp = by_month.pivot(index='month', columns='stnNm', values='temp')
print(temp.loc[[1, 2, 7, 8, 12]].round(1))
gap = (temp['부산'] - temp['서울']).round(1)   # 차를 낸 뒤 반올림
print('부산 − 서울:', gap.loc[[1, 2, 12]].to_dict())
wettest = by_month.groupby('stnNm')['rain'].idxmax()
print(by_month.loc[wettest].round(1))
yearly = two.groupby('stnNm')['sumRn'].sum().round(1)
print('연 강수량:', yearly.to_dict())

In [ ]:
# C10 | 일교차 분포: 하루 최고 − 최저 기온을 박스 플롯으로 견줍니다.
two['range'] = two['maxTa'] - two['minTa']
fig, ax = plt.subplots(figsize=(6, 2.0), layout='constrained')
sns.boxplot(data=two, x='range', y='stnNm', order=['서울', '부산'],
            hue='stnNm', palette=city_colors, legend=False,
            width=0.5, ax=ax)
ax.set(xlabel='일교차 (℃)', ylabel='')
print(two.groupby('stnNm')['range'].median())
plt.show()

### 직접 해 보기

**W12-02** 월 대신 계절로 강수량을 비교합니다.

1. 3~5월은 봄, 6~8월은 여름, 9~11월은 가을, 같은 해의 12·1·2월은 겨울로 정한 사전을 만들고 `two['month'].map(사전)`(값을 사전의 값으로 바꿔 끼움)으로 계절 열을 더합니다.
2. 계절·도시별 강수량 합계를 출력하고, 7장 C09의 `reindex()`로 봄 → 겨울 순서를 고정한 뒤, 계절을 0~3 위치에 두고 두 도시의 막대를 위 C09처럼 나란히 그립니다(눈금에 계절 이름). 부산 막대에는 빗금을 넣습니다.
3. `ch12_output/W12-02.png`로 저장해 다시 엽니다(폴더가 없으면 2장 C05처럼 만듭니다).

**완료 기준**: 도시마다 네 계절 합계를 더한 값이 C09B의 연 강수량과 같고, 막대가 봄 → 겨울 순서이며, 부산 막대에 빗금이 있다.

In [ ]:
# W12-02 — 여기에서 직접 해 봅니다.

### 직접 해 보기

**W12-03** 일평균 기온과 평균 상대습도의 관계를 봅니다.

1. `two`로 X축 `avgTa`, Y축 `avgRhm`인 산점도를 그리고 `hue`·`style`로 도시를 나눕니다(5.2절).
2. 축 라벨에 단위(℃·%)를 적고 출처 줄을 답니다.
3. 도시나 기온 구간의 분포를 한 문장으로 쓰고 원자료 값 한 곳과 대조합니다.

**완료 기준**: 단위·도시 구분이 있고, 구체적인 관찰을 원자료와 대조했으며 원인을 단정하지 않는다.

In [ ]:
# W12-03 — 여기에서 직접 해 봅니다.

## 12.5 실패 메시지와 AI 초안 검수

**키가 없거나 C05가 오류로 멈추면** C01–C05와 C11은 지면의 출력으로 읽고, C06부터 예제 저장소의 저장본(받아 둔 CSV, 12.3절)으로 실행합니다.

In [ ]:
# C11 | AI 초안: 검수 연습을 위해 재구성한 초안입니다.
p = {'serviceKey': key, 'dataType': 'JSON', 'dataCd': 'ASOS',
     'dateCd': 'DAY', 'startDt': '20250101', 'endDt': '20251231',
     'stnIds': '108'}
r = requests.get(URL, params=p)
draft = pd.DataFrame(r.json()['response']['body']['items']['item'])
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
ax.plot(draft['tm'], draft['avgTa'])
ax.set(title='서울 2025년 일평균 기온', ylabel='기온 (℃)')
print(len(draft), '행')
plt.show()

### 직접 해 보기

**W12-04** C11을 고쳐 다시 실행합니다.

1. 위 검수 순서로 찾은 문제를 고칩니다. C05가 성공했으면 받은 `weather`에서, 키가 없거나 C05가 실패했으면 C06의 저장본에서 서울을 골라 그립니다. 고칠 곳은 C11에 주석으로 적습니다.
2. 고친 그림을 `ch12_output/W12-04.png`로 저장해 다시 엽니다.
3. 고친 곳과 확인한 값을 적습니다.

**완료 기준**: 코드에 키가 없고, C05가 성공했으면 받은 행 수가 `totalCount`와 같으며(키가 없거나 C05가 실패했으면 고칠 곳을 검수 순서대로 적었으며), 기온 열이 `float64`이고, 그림이 1년 365일을 날짜 X축과 숫자 Y축으로 보인다.

In [ ]:
# W12-04 — 여기에서 직접 해 봅니다.

## 12.6 앱으로 잇기

지면의 번호 없는 코드(앱·배포 파일에 넣는 조각이나 지면이 말한 자리에서 실행하는 확인 코드 — 참고용):

```python
params['serviceKey'] = unquote(st.secrets['DATA_GO_KR_KEY'].strip())


@st.cache_data(ttl=3600)        # 같은 입력은 한 시간까지 재사용
def load_weather(stn):
    return fetch_daily(stn)     # C05의 확인·형식 바꾸기 그대로
```

## 12.7 해석과 점검

## 직접 해 보기 풀이

먼저 위의 연습 칸에서 직접 해 본 뒤에 봅니다. 풀이는 한 가지 예이고, 완료 기준을 만족하면 다른 코드도 됩니다.

### W12-01

In [ ]:
# W12-01 | 풀이: 강릉의 행 수·기간을 보고 강수량이 0보다 큰 날과 그렇지 않은 날을 셉니다.
city = weather[weather['stnNm'] == '강릉']
print(city['tm'].agg(['size', 'min', 'max']))
rain_days = (city['sumRn'] > 0).sum()
other_days = (city['sumRn'] <= 0).sum()
print('일강수량이 0보다 큰 날', rain_days, '· 그렇지 않은 날', other_days)
assert rain_days + other_days == len(city)

In [ ]:
# 3. 일강수량이 가장 많은 날
wettest = city.loc[city['sumRn'].idxmax(), ['tm', 'sumRn']]
print(wettest['tm'].date(), wettest['sumRn'], 'mm')

### W12-02

In [ ]:
# W12-02 | 풀이: 계절 사전으로 계절 열을 더하고 계절별 강수량 합계를 그립니다.
season_of = {3: '봄', 4: '봄', 5: '봄', 6: '여름', 7: '여름', 8: '여름',
             9: '가을', 10: '가을', 11: '가을', 12: '겨울', 1: '겨울', 2: '겨울'}
two['season'] = two['month'].map(season_of)
order = ['봄', '여름', '가을', '겨울']
season = (two.groupby(['season', 'stnNm'])['sumRn'].sum()
          .unstack()[['서울', '부산']].reindex(order))   # 봄 → 겨울 순서
print(season.round(1))
print('계절 합 = 연 합:', season.sum().round(1).to_dict())   # C09B와 대조
fig, ax = plt.subplots(figsize=(6, 2.6), layout='constrained')
for i, city in enumerate(['서울', '부산']):
    ax.bar([k - 0.2 + 0.4 * i for k in range(4)], season[city], width=0.4,
           color=city_colors[city], hatch=['', '//'][i],
           edgecolor='black', linewidth=0, label=city)
ax.set(xticks=range(4), xticklabels=order, ylabel='강수량 (mm)',
       title='2025년 계절별 강수량')
ax.legend()
Path('ch12_output').mkdir(exist_ok=True)
fig.savefig('ch12_output/W12-02.png', dpi=150)
plt.show()
# 저장한 ch12_output/W12-02.png를 파일 탐색기에서 다시 엽니다.
# 제목·축·범례가 잘리지 않았는지, 파일 이름이 맞는지 확인합니다.

### W12-03

In [ ]:
# W12-03 | 풀이: 일평균 기온과 평균 습도의 산점도를 도시마다 색과 모양으로 나눕니다.
fig, ax = plt.subplots(figsize=(6, 2.8), layout='constrained')
sns.scatterplot(data=two, x='avgTa', y='avgRhm', hue='stnNm', style='stnNm',
                hue_order=['서울', '부산'], style_order=['서울', '부산'],
                palette=city_colors, s=14, alpha=0.6, ax=ax)
ax.set(xlabel='일평균 기온 (℃)', ylabel='평균 상대습도 (%)',
       title='2025년 일평균 기온과 습도')
ax.legend(title=None)
ax.text(0.01, 0.03, source, transform=ax.transAxes, fontsize=9, color='#666666')
plt.show()
# 3.(예) 두 도시 모두 기온이 높은 날일수록 습도가 높은 쪽에 점이 모이고, 영하인 날의 습도는 부산이 서울보다 낮다.
# 원자료 대조: 서울 2025-01-03의 일평균 기온은 -1.1℃, 평균 상대습도는 52.0%입니다.

### W12-04

In [ ]:
# W12-04 | 풀이: 검증한 weather에서 서울을 골라 1년 전체를 그립니다.
seoul = weather[weather['stnNm'] == '서울']  # C05의 받은 표 또는 C06의 저장본
print(len(seoul), '행 ·', seoul['avgTa'].dtype)
fig, ax = plt.subplots(figsize=(6, 2.4), layout='constrained')
ax.plot(seoul['tm'], seoul['avgTa'], color=city_colors['서울'], linewidth=1)
ax.axhline(0, color='#666666', linewidth=0.8)
ax.set(title='서울 2025년 일평균 기온', ylabel='기온 (℃)')
ax.text(0.01, 0.03, source, transform=ax.transAxes, fontsize=9, color='#666666')
Path('ch12_output').mkdir(exist_ok=True)
fig.savefig('ch12_output/W12-04.png', dpi=150)
plt.show()
# 3.(예) 고친 곳: 키는 P07의 key(코드에 적지 않음), numOfRows=366, timeout=10, 상태 코드·결과 코드·totalCount 확인,
#   숫자·날짜 변환(모두 fetch_daily()). 확인한 값: 365행, avgTa float64, X축이 1~12월.
# 저장한 ch12_output/W12-04.png를 파일 탐색기에서 다시 엽니다.
# 제목·축·출처가 잘리지 않았는지, 파일 이름이 맞는지 확인합니다.